In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [6]:
df = pd.read_csv("train_clean.csv")

In [7]:
df = df.drop('id', axis=1)
X = df.drop("Will_Buy_EV", axis=1)
y = df["Will_Buy_EV"]

num_col = []
cat_col = []

for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        num_col.append(col)
    else:
        cat_col.append(col)

for col in cat_col:
    X[col] = X[col].astype('category')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
import optuna
import lightgbm as lgb

# 1. Split data training khusus untuk validasi Optuna (Holdout) agar sangat cepat
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

def objective(trial):
    # 2. Search space parameter krusial saja
    param = {
        'n_estimators': 1500, # Set tinggi, dikontrol oleh early stopping
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 256),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 300),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': 2,
        'class_weight': 'balanced'
    }

    model = lgb.LGBMClassifier(**param)

    # 3. Fit dengan Early Stopping
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(stopping_rounds=30, verbose=False),
            # lgb.log_evaluation(0) # Buka comment ini jika ada warning output terlalu banyak
        ]
    )

    # 4. Evaluasi ROC-AUC
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred_proba)

# 5. Eksekusi Optuna dengan Pruner (menghentikan trial buruk lebih awal)
study = optuna.create_study(
    direction='maximize', 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)
study.optimize(objective, n_trials=30) # 30 trial cukup untuk awal

print("Parameter Terbaik:", study.best_params)
print("ROC-AUC Tertinggi di Optuna:", study.best_value)

[I 2026-09-13 18:36:03,746] A new study created in memory with name: no-name-e3eb8764-3f57-4d37-8f0e-f444a324de5e


[LightGBM] [Info] Number of positive: 74738, number of negative: 353207
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.177300
[LightGBM] [Debug] init for col-wise cost 0.001373 seconds, init for row-wise cost 0.080816 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.127203 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 656
[LightGBM] [Info] Number of data points in the train set: 427945, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Debug] Trained a tree with leaves = 234 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 234 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 234 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 234 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 234 and depth = 9


[I 2026-09-13 18:37:29,807] Trial 0 finished with value: 0.9404035783806063 and parameters: {'learning_rate': 0.022972110515248386, 'num_leaves': 234, 'max_depth': 9, 'min_child_samples': 48, 'subsample': 0.9447992679791862, 'colsample_bytree': 0.9776952611791867}. Best is trial 0 with value: 0.9404035783806063.


[LightGBM] [Info] Number of positive: 74738, number of negative: 353207
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.177300
[LightGBM] [Debug] init for col-wise cost 0.000014 seconds, init for row-wise cost 0.063407 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073363 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 656
[LightGBM] [Info] Number of data points in the train set: 427945, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Debug] Trained a tree with leaves = 180 and depth = 13
[LightGBM] [Debug] Trained a tree with leaves = 180 and depth = 13
[LightGBM] [Debug] Trained a tree with leaves = 180 and depth = 13
[LightGBM] [Debug] Trained a tree with leaves = 180 and depth = 13
[LightGBM] [Debug] Trained a tree with leaves = 180 and depth 

[I 2026-09-13 18:37:42,096] Trial 1 finished with value: 0.9406291143779346 and parameters: {'learning_rate': 0.18950121299768824, 'num_leaves': 180, 'max_depth': 13, 'min_child_samples': 68, 'subsample': 0.8588331141282037, 'colsample_bytree': 0.7023057679686291}. Best is trial 1 with value: 0.9406291143779346.


...

[LightGBM] [Info] Number of positive: 74738, number of negative: 353207
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.177300
[LightGBM] [Debug] init for col-wise cost 0.000016 seconds, init for row-wise cost 0.029654 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.039961 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 656
[LightGBM] [Info] Number of data points in the train set: 427945, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Debug] Trained a tree with leaves = 203 and depth = 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 160 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 203 and depth = 8
[LightGBM] [Warning] No further splits with positive gain

[I 2026-09-13 18:52:57,692] Trial 28 finished with value: 0.9408948922563878 and parameters: {'learning_rate': 0.15740405709578434, 'num_leaves': 203, 'max_depth': 8, 'min_child_samples': 169, 'subsample': 0.8687069270203945, 'colsample_bytree': 0.6579277594195362}. Best is trial 15 with value: 0.941236440040376.


[LightGBM] [Info] Number of positive: 74738, number of negative: 353207
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.177300
[LightGBM] [Debug] init for col-wise cost 0.000013 seconds, init for row-wise cost 0.035123 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044319 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 656
[LightGBM] [Info] Number of data points in the train set: 427945, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 120 and depth = 7
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 95 and depth = 7
[LightGBM] [Warning] No further splits with posit

[I 2026-09-13 18:53:20,026] Trial 29 finished with value: 0.9410053198426177 and parameters: {'learning_rate': 0.09908833191570443, 'num_leaves': 165, 'max_depth': 7, 'min_child_samples': 213, 'subsample': 0.9277575017199523, 'colsample_bytree': 0.6750629394587345}. Best is trial 15 with value: 0.941236440040376.


Parameter Terbaik: {'learning_rate': 0.11890912626024237, 'num_leaves': 35, 'max_depth': 5, 'min_child_samples': 171, 'subsample': 0.8227983632960099, 'colsample_bytree': 0.6102452703257967}
ROC-AUC Tertinggi di Optuna: 0.941236440040376
